# 07: Final evaluation and saved models

The 2024/25 validation season was used to choose the model: Poisson regression on recent-form features. This notebook touches the untouched 2025/26 test season **once**, reports the result, then refits the chosen model on every completed season and saves it for the demo.

**Contents**
1. Refit the chosen model on the training seasons (up to 2023/24)
2. Evaluate once on the 2025/26 test season against the same four benchmarks
3. Refit on all completed seasons (through 2025/26) and save with joblib
4. Smoke-test the saved artifact on a fixture

## 1. Refit the chosen model

All reusable functions now live in `src/football_prediction/`. `result_metrics` is the fixed log-loss helper: it computes the probability of the true result by hand, so the H/D/A column order is respected (scikit-learn's `log_loss` would sort the labels as A/D/H and silently swap home and away).

In [1]:
import numpy as np
import pandas as pd

from football_prediction import (
    FEATURES, LABELS, load_model_table, load_team_form, split,
    fit_models, predict_goals, save_models, load_models,
    result_metrics, goal_metrics, implied_probs,
    scoreline_grid, result_probs, predict_match,
)

df = load_model_table()
s = split(df)
train, valid, test = s["train"], s["valid"], s["test"]
print("train / validation / test:", len(train), len(valid), len(test))

models = fit_models(train)                      # chosen on 2024/25, not on the test season
lam_h, lam_a = predict_goals(models, test)
print("mean predicted goals, home/away:", round(lam_h.mean(), 3), round(lam_a.mean(), 3))
print("mean actual goals, home/away:", round(test["home_goals"].mean(), 3), round(test["away_goals"].mean(), 3))

train / validation / test: 2560 375 375
mean predicted goals, home/away: 1.535 1.271
mean actual goals, home/away: 1.525 1.235


## 2. One final evaluation on 2025/26

The same four contenders as in notebook 06, scored on the same matches (those with Bet365 odds): the training result shares, the training-average goals pushed through the Poisson grid, the chosen Poisson regression, and the margin-free bookmaker probabilities. Accuracy and log loss are computed by `result_metrics`; goal-level MAE and Poisson deviance by `goal_metrics`.

In [2]:
mask = test[["B365H", "B365D", "B365A"]].notna().all(axis=1)
v = test[mask]

p_const = np.tile(train["FTR"].value_counts(normalize=True)[LABELS].to_numpy(), (len(v), 1))
p_base = result_probs(np.full(len(v), train["home_goals"].mean()),
                      np.full(len(v), train["away_goals"].mean()))
p_model = result_probs(lam_h[mask.to_numpy()], lam_a[mask.to_numpy()])
p_book = implied_probs(v[["B365H", "B365D", "B365A"]].to_numpy())

table = pd.DataFrame([
    result_metrics(n, p, v["FTR"])
    for n, p in [("Constant (training result shares)", p_const),
                 ("Baseline goals -> Poisson", p_base),
                 ("Poisson regression", p_model),
                 ("Bookmaker odds (Bet365)", p_book)]
]).round(3)
table

,model,accuracy,log loss
0,Constant (training result shares),0.424,1.086
1,Baseline goals -> Poisson,0.424,1.081
2,Poisson regression,0.488,1.046
3,Bookmaker odds (Bet365),0.493,1.016


In [3]:
goal_table = pd.DataFrame([
    goal_metrics("Baseline goals", np.full(len(test), train["home_goals"].mean()),
                 np.full(len(test), train["away_goals"].mean()), test),
    goal_metrics("Poisson regression", lam_h, lam_a, test),
]).round(3)
goal_table

,model,MAE home,MAE away,Poisson dev home,Poisson dev away
0,Baseline goals,0.998,0.883,1.122,1.117
1,Poisson regression,0.975,0.845,1.095,1.075


In [4]:
# Exact-score hit rate and calibration on the test season
grid = scoreline_grid(lam_h, lam_a)
best = grid.reshape(len(grid), -1).argmax(axis=1)
ph, pa = np.divmod(best, 11)
exact = ((ph == test["home_goals"].to_numpy()) & (pa == test["away_goals"].to_numpy())).mean()
print(f"Exact scoreline hit rate: {exact:.1%}")

calib = pd.DataFrame({
    "predicted share": result_probs(lam_h, lam_a).mean(axis=0),
    "actual share": test["FTR"].value_counts(normalize=True)[LABELS],
}, index=LABELS).round(3)
calib

Exact scoreline hit rate: 10.4%


,predicted share,actual share
H,0.436,0.424
D,0.243,0.272
A,0.321,0.304


## 3. Refit on every completed season and save

The test season has been scored, so it can now be used for training. Seasons 1718 to 2526 are completed; the in-progress 2026/27 season stays out of the fit. Both pipelines and their metadata are saved in one joblib artifact in `models/` (gitignored, so regenerate it with this cell).

In [5]:
final_train = df.query("Season <= '2526'")
print("matches used for the final fit:", len(final_train))
final_models = fit_models(final_train)
path = save_models(final_models, trained_on="1718-2526")
print("saved:", path)

matches used for the final fit: 3310
saved: /Users/mac/football-prediction/models/poisson.joblib


In [6]:
# Smoke-test the saved artifact: predict a fixture using the latest available form
artifact = load_models()
print("artifact keys:", sorted(artifact), "| trained on:", artifact["trained_on"])
team_form = load_team_form()
pred = predict_match("Man City", "Chelsea", team_form["Date"].max(), artifact, team_form)
print(f"{pred['home']} vs {pred['away']} | expected goals {pred['lam_h']:.2f} - {pred['lam_a']:.2f}")
print({k: round(float(p), 3) for k, p in pred["probs"].items()}, "| predicted score:", pred["score"])

artifact keys: ['away', 'features', 'home', 'trained_on'] | trained on: 1718-2526
Man City vs Chelsea | expected goals 1.83 - 1.23
{'H': 0.513, 'D': 0.229, 'A': 0.258} | predicted score: (2, 1)


## 4. Summary

**2025/26 test season (375 matches, models trained only on seasons up to 2023/24)**

| Model | Accuracy | Log loss |
|---|---|---|
| Constant (training result shares) | 42.4% | 1.086 |
| Baseline goals -> Poisson | 42.4% | 1.081 |
| **Poisson regression** | **48.8%** | **1.046** |
| Bookmaker odds (Bet365) | 49.3% | 1.016 |

The chosen model holds up on the untouched season. It beats both baselines by about 3.3% on log loss versus the goals baseline (1.081 -> 1.046) and gains 6.4 accuracy points, closing roughly 54% of the log-loss gap to the bookmakers. It is only 0.5 accuracy points behind the bookmakers this time, well within the roughly 2.6-point sampling noise of a 375-match season, but the bookmakers still have the lower log loss (1.016 against 1.046), so they remain the stronger probabilistic benchmark.

**Goals.** MAE falls from 0.998 to 0.975 for home goals and from 0.883 to 0.845 for away goals; Poisson deviance falls from 1.122 to 1.095 and from 1.117 to 1.075. The improvement is smaller than on validation but consistent in direction on every goal metric.

**Exact scores and calibration.** The most likely exact score was right in 10.4% of matches (12.3% on validation), which is close to what always naming 1-1 would achieve, so it is not a measure of skill. Predicted result shares were 43.6% home / 24.3% draw / 32.1% away against actual shares of 42.4% / 27.2% / 30.4%. Home wins are well calibrated here; draws are under-predicted by about 3 points, the opposite of the validation season, which is a reminder that one season is a small sample.

**Saved artifact.** After the test score was recorded, both models were refitted on all 3,310 completed-season matches with history (seasons 1718 to 2526) and saved with joblib to `models/poisson.joblib`. The in-progress 2026/27 season is still excluded. The smoke test predicts Man City 1.83 - 1.23 Chelsea from the latest available form, with 51.3% / 22.9% / 25.8% for home win / draw / away win.